# ⚡ Módulo 12 - Notebook 02: Operaciones de Columnas y Functions

## 📑 Filtrado, Selección y Manipulación Avanzada

**Libro:** Saliendo de lo Pandito  
**Módulo:** 12 - PySpark Transformación Avanzada  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Filtrar** datos con condiciones complejas  
✅ **Seleccionar** y renombrar columnas eficientemente  
✅ **Aplicar** funciones de agregación y ventana  
✅ **Usar** expresiones SQL en DataFrames  
✅ **Optimizar** operaciones de columnas

---

## 📋 Pre-requisitos

* ✅ Notebook 12_01 completado (Transformaciones básicas)
* ✅ Conocimiento de withColumn()
* ✅ Familiaridad con pyspark.sql.functions

---

## 📚 Contenido

1. Filtrado Avanzado (filter/where)
2. Selección y Renombrado de Columnas
3. Expresiones SQL en DataFrames
4. Funciones de Agregación
5. Funciones de Ventana (Window Functions)
6. Caso Integrador: Análisis de Ventas Complejo

---

## 💡 Por qué importa

**Operaciones de columnas son fundamentales:**

* 🔍 **Filtrado:** Reducir volumen de datos procesados
* 📊 **Agregación:** Resumir millones de registros
* 📝 **Expresiones SQL:** Aprovechar conocimiento SQL
* ⚡ **Optimización:** Predicate pushdown automático

**La base de todo pipeline ETL**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros totales: {df.count():,}")
    print(f"   🏛️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Mostrar muestra de datos
    print(f"\n📊 Muestra de datos (primeras 5 filas):")
    df.show(5, truncate=False)
    
    print(f"\n🎯 Este notebook aplicará:")
    print(f"   • Filtrado complejo con múltiples condiciones")
    print(f"   • Selección y renombrado de columnas")
    print(f"   • Agregaciones por grupos")
    print(f"   • Funciones de ventana (ranking, cumulative)")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Operaciones Avanzadas de Columnas

### 🔍 Filtrado Avanzado (filter / where)

**Sintaxis:**
```python
# Ambos son equivalentes
df.filter(condicion)
df.where(condicion)
```

**Formas de expresar condiciones:**

**1️⃣ String SQL:**
```python
df.filter("ventas > 100000")
df.filter("ventas > 100000 AND zona = 'Centro'")
```

**2️⃣ Expresión con F.col():**
```python
df.filter(F.col("ventas") > 100000)
df.filter((F.col("ventas") > 100000) & (F.col("zona") == "Centro"))
```

**3️⃣ Múltiples condiciones:**
```python
# AND: &
df.filter((F.col("ventas") > 100000) & (F.col("margen") > 0.20))

# OR: |
df.filter((F.col("zona") == "Centro") | (F.col("zona") == "Norte"))

# NOT: ~
df.filter(~F.col("zona").isin(["Sur", "Este"]))
```

**⚠️ IMPORTANTE:** Usar `&` y `|`, NO `and` y `or` de Python.

---

### 📑 Selección y Renombrado de Columnas

**Seleccionar columnas:**
```python
# Método 1: Lista de strings
df.select("sucursal", "ventas", "fecha")

# Método 2: F.col()
df.select(F.col("sucursal"), F.col("ventas"))

# Método 3: Con transformaciones
df.select(
    F.col("sucursal"),
    (F.col("ventas") * 1.21).alias("ventas_con_iva")
)
```

**Renombrar columnas:**
```python
# Método 1: alias()
df.select(
    F.col("sucursal").alias("tienda"),
    F.col("ventas").alias("monto")
)

# Método 2: withColumnRenamed()
df.withColumnRenamed("sucursal", "tienda")

# Método 3: Renombrar múltiples
df.toDF("col1", "col2", "col3")  # Renombra todas
```

**Drop columnas:**
```python
df.drop("columna_innecesaria", "otra_columna")
```

---

### 📊 Agregaciones

**Agregación simple:**
```python
df.agg(
    F.sum("ventas").alias("ventas_totales"),
    F.avg("ventas").alias("ventas_promedio"),
    F.count("*").alias("cantidad_registros")
)
```

**Agregación por grupo:**
```python
df.groupBy("zona").agg(
    F.sum("ventas").alias("ventas_totales"),
    F.avg("ventas").alias("ventas_promedio"),
    F.count("*").alias("sucursales")
)
```

**Múltiples dimensiones:**
```python
df.groupBy("zona", "mes").agg(
    F.sum("ventas").alias("ventas_totales")
).orderBy("zona", "mes")
```

---

### 📝 Expresiones SQL en DataFrames

**selectExpr() - SQL directo:**
```python
df.selectExpr(
    "sucursal",
    "ventas * 1.21 AS ventas_con_iva",
    "CASE WHEN ventas > 100000 THEN 'Alto' ELSE 'Bajo' END AS categoria"
)
```

**expr() - Expresiones complejas:**
```python
from pyspark.sql.functions import expr

df.withColumn("margen_pct", expr("(ventas - costo) / ventas * 100"))
```

---

### 📊 Funciones de Ventana (Window Functions)

**Concepto:** Operaciones sobre un "grupo" de filas relacionadas.

**Definir ventana:**
```python
from pyspark.sql.window import Window

# Ventana particionada por zona, ordenada por ventas
window_spec = Window.partitionBy("zona").orderBy(F.desc("ventas"))
```

**Ranking:**
```python
df.withColumn(
    "ranking_por_zona",
    F.row_number().over(window_spec)
)
```

**Acumulados:**
```python
df.withColumn(
    "ventas_acumuladas",
    F.sum("ventas").over(
        Window.partitionBy("zona").orderBy("fecha")
    )
)
```

**Lag/Lead (valores previos/siguientes):**
```python
# Ventas del mes anterior
df.withColumn(
    "ventas_mes_anterior",
    F.lag("ventas", 1).over(
        Window.partitionBy("sucursal").orderBy("fecha")
    )
)

# Calcular variación
df.withColumn(
    "variacion",
    F.col("ventas") - F.col("ventas_mes_anterior")
)
```

---

### 💼 Caso de Uso: Top 3 Sucursales por Zona

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Agregar ventas por sucursal
df_agg = df.groupBy("zona", "sucursal").agg(
    F.sum("ventas").alias("ventas_totales")
)

# 2. Definir ventana por zona
window = Window.partitionBy("zona").orderBy(F.desc("ventas_totales"))

# 3. Agregar ranking
df_ranked = df_agg.withColumn("ranking", F.row_number().over(window))

# 4. Filtrar top 3
top3 = df_ranked.filter(F.col("ranking") <= 3)

top3.show()
```

**Resultado:**
```
+-------+-----------+--------------+-------+
|  zona |  sucursal |ventas_totales|ranking|
+-------+-----------+--------------+-------+
|Centro |Sucursal A |     5,500,000|      1|
|Centro |Sucursal B |     4,200,000|      2|
|Centro |Sucursal C |     3,800,000|      3|
| Norte |Sucursal D |     4,100,000|      1|
| Norte |Sucursal E |     3,900,000|      2|
| Norte |Sucursal F |     3,500,000|      3|
+-------+-----------+--------------+-------+
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("📑 OPERACIONES DE COLUMNAS Y FUNCTIONS")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • filter/where - Filtrado avanzado")
print("  • select/selectExpr - Selección y proyección")
print("  • groupBy + agg - Agregaciones")
print("  • Window Functions - Ranking y acumulados")

print("\n📖 Funciones clave:")
print("  - df.filter(condicion)")
print("  - df.select(F.col('col').alias('nuevo'))")
print("  - df.groupBy('col').agg(F.sum('val'))")
print("  - Window.partitionBy('col').orderBy('val')")
print("  - F.row_number(), F.rank(), F.lag(), F.lead()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')